# Entry-idea opportunity-count aggregates

**Status:** Descriptive cached-context count, not a trade or model-performance result. **Reproduction:** R2 for the four family-level annual episode totals and their plot, recomputed from the tracked per-idea/per-timeframe CSV. **Tier:** A, CPU.

This notebook checks the annual family totals against the per-timeframe machine-readable evidence and regenerates the comparison plot. The original feature cache is not tracked, so it cannot independently recount market bars or verify the feature extraction from raw data. Counts describe cached context before session eligibility, policy, execution, or account-risk filters.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").is_file():
    ROOT = ROOT.parent
EVIDENCE = ROOT / "evidence/research/entry_idea_opportunity_counts"
annual = pd.read_csv(EVIDENCE / "annual_summary.csv")
by_timeframe = pd.read_csv(EVIDENCE / "by_idea_timeframe.csv")
print("annual rows:", len(annual), "per-idea/timeframe rows:", len(by_timeframe))
print("years:", annual["year"].tolist())
print("families:", sorted(by_timeframe["idea"].unique()))

In [ ]:
family_columns = {
    "PO3": "po3_signal_episode_starts",
    "Order block": "order_block_signal_episode_starts",
    "Standard deviation": "standard_deviation_signal_episode_starts",
    "FVG/IFVG reaction": "fvg_ifvg_reaction_signal_episode_starts",
}
recomputed = (
    by_timeframe.groupby(["year", "idea"], as_index=False)
    .agg(
        active_m1_bars=("active_m1_bars", "sum"),
        episode_starts=("episode_starts", "sum"),
        both_sides_active_m1_bars=("both_sides_active_m1_bars", "sum"),
    )
    .sort_values(["year", "idea"])
    .reset_index(drop=True)
)

for _, annual_row in annual.iterrows():
    year = int(annual_row["year"])
    year_rows = recomputed.loc[recomputed["year"].eq(year)]
    for idea, annual_column in family_columns.items():
        match = year_rows.loc[year_rows["idea"].eq(idea)]
        assert len(match) == 1, (year, idea)
        assert int(match.iloc[0]["episode_starts"]) == int(annual_row[annual_column])
    assert int(year_rows["episode_starts"].sum()) == int(
        annual_row["idea_timeframe_episode_starts_sum"]
    )
    assert int(year_rows["active_m1_bars"].sum()) == int(
        annual_row["idea_timeframe_active_bars_sum"]
    )
    assert int(year_rows["both_sides_active_m1_bars"].sum()) == int(
        annual_row["idea_timeframe_both_sides_bars_sum"]
    )

print("Per-family episode counts and timeframe aggregates match annual_summary.csv.")
recomputed

## Regenerated annual episode-count figure

Each bar sums episode starts across the reported source timeframes. PO3 is already provided once as `Any TF`; do not interpret these counts as trades, unique opportunities, or independent observations. The partially exposed 2026 calendar is shown as descriptive historical data, not as an untouched holdout.

In [ ]:
plot_data = recomputed.pivot(index="idea", columns="year", values="episode_starts").reindex(
    list(family_columns)
)
ax = plot_data.plot(kind="bar", figsize=(9, 4), color=["#627d98", "#d07b45", "#6f9c76"])
ax.set_title("Cached entry-idea episode starts by year")
ax.set_ylabel("Episode starts across reported source timeframes")
ax.set_xlabel("Idea family")
ax.tick_params(axis="x", rotation=20)
ax.legend(title="Calendar year")
plt.tight_layout()
plt.show()

## Claim boundary

R2 applies to the family-level annual episode totals, the timeframe-level active-bar/episode tables, and this regenerated visualization from the two tracked CSVs. The separate annual `unique_bars_*` counts remain inspectable in `annual_summary.csv` but are not recomputed here because the timestamp-level feature cache is unavailable. No count is an executable trade count or evidence of policy performance. This package does not support R3 frozen-policy replay or R4 retraining.

See the [source report](https://github.com/nntk072/Quantitative-RL-Trading-System/blob/main/evidence/research/entry_idea_opportunity_counts/entry_idea_opportunity_counts.md), [multi-timeframe preview limitations](../research/order_block_and_multitimeframe_context.md), and [result registry](result_registry.md).